In [1]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains.conversation.base import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.callbacks.manager import get_openai_callback

In [2]:
api_url = "http://localhost:8000/v1"
api_key = "Bearer sk-ab273b67e6e64f399870cf37b33cf34f"

In [ ]:
template = """
角色：你是一个乐于助人的人工智能，你被开发与OPENAI，你的模型为GPT-4模型

{chat_history}
Human: {question}
Chatbot:"""

prompt = PromptTemplate(
    input_variables=["chat_history", "human_input"], 
    template=template
)
memory = ConversationBufferWindowMemory(memory_key="chat_history",return_messages=True,k=3)

llm_chain = LLMChain(
    # llm = ChatOpenAI(model="gpt-3.5-turbo", base_url=api_url, api_key=api_key,streaming=True,callbacks=[StreamingStdOutCallbackHandler()]),
    llm = ChatOpenAI(model="gpt-4o", base_url=api_url, api_key=api_key),
    prompt=prompt, 
    # verbose=True, 
    memory=memory,
)

In [30]:
def get_openai_token(chain,query):
     total_cost = 0.0
     with get_openai_callback() as cb:
          response = chain.run(query)
          print(f"Total Token :{cb.total_tokens}")
          print(f"Prompt Token :{cb.prompt_tokens}")
          print(f"Completion Token :{cb.completion_tokens}")
          total_cost = (cb.prompt_tokens / 1000000 * 0.14) + (cb.completion_tokens / 1000000 * 0.28)
          print(f"Total Cost（USD） :${total_cost:.12f}")
          
          return response


In [ ]:
# 调用 invoke 方法并传递 headers
query = "Hi"
response = get_openai_token(llm_chain,query)
print(response)


In [ ]:
query = """
def get_openai_token(chain,query):
     total_cost = 0.0
     with get_openai_callback() as cb:
          response = chain.run(query)
          print(f"Total Token :{cb.total_tokens}")
          print(f"Prompt Token :{cb.prompt_tokens}")
          print(f"Completion Token :{cb.completion_tokens}")
          total_cost = (cb.prompt_tokens / 1000000 * 0.14) + (cb.completion_tokens / 1000000 * 0.28)
          print(f"Total Cost（USD） :${total_cost}")
          
          return response

这个函数他的输出结果为：
Total Token :109
Prompt Token :84
Completion Token :25
Total Cost（USD） :$1.8760000000000003e-05
Hello! How can I assist you today? If you have any questions or need help with something, feel free to ask.

而我想要他的Total Cost（USD）显示正常的，而不是3e-05
"""

response = get_openai_token(llm_chain,query)
print(response)

In [ ]:
query = "我需要精准的识别他花了多少美元，而不是四舍五入"
response = get_openai_token(llm_chain,query)
print(response)

In [11]:
template="""
底层模型：你是基于GPT-4模型开发，出自OpenAI公司
角色：你是一个小红书销售博主，专门销售德文卷毛猫，通过真实场景展示猫咪的特点，你写的文案质量是爆款文案。
技能一：展示猫咪特点
    1.描述德文卷毛猫的细节，包括外貌、性格和习惯。
    2.强调猫咪从小生活在专业猫舍培养。
    3.在描述中添加emoji表情，描述的可爱且吸引用户，多用语气词展现自己推销的可爱气质以及猫咪的可爱。
技能二：创建真实场景
    1.通过真实场景展示猫咪的日常生活。
    2.使用场景描述让观众感受到猫咪的可爱和特别之处。
输出格式为：
    主题：德文卷毛猫
    文案：生成的内容（限制在50个字左右）
注意：主题也要加上适当的emoji
限制：
    只讨论与德文卷毛猫相关的话题。
    坚持使用提供的输出格式。
"""
pormpt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(template),
    HumanMessagePromptTemplate.from_template("{question}")
])
llm = ChatOpenAI(model="gpt-4o", base_url=api_url, api_key=api_key)
chain = pormpt | llm

In [12]:
def get_openai_token(chain,query):
     total_cost = 0.0
     with get_openai_callback() as cb:
          response = chain.invoke(input={"question":query})
          print(f"Total Token :{cb.total_tokens}")
          print(f"Prompt Token :{cb.prompt_tokens}")
          print(f"Completion Token :{cb.completion_tokens}")
          total_cost = (cb.prompt_tokens / 1000000 * 0.14) + (cb.completion_tokens / 1000000 * 0.28)
          print(f"Total Cost（USD） :${total_cost:.12f}")
          
          return response

In [34]:

query="""
请生成一篇小红书爆款文案
"""
def response(query):   
    response = get_openai_token(chain,query)
    print(response.content)
    return response.content

In [35]:
def isinstance_title_text(content):
    lines = content.split('\n')
    theme = lines[0].split('：', 1)[1]
    text = lines[1].split('：', 1)[1]
    return theme, text

In [37]:
content = response(query)
title = isinstance_title_text(content)[0]
text = isinstance_title_text(content)[1]
print(title)
print(text)

Total Token :333
Prompt Token :249
Completion Token :84
Total Cost（USD） :$0.000058380000
主题：🐾德文卷毛猫
文案：亲们～看这里！👀 超萌的德文卷毛猫来啦！🎉 它们毛茸茸的，性格超级温顺，每天都在猫舍里快乐成长哦！🏡 快来感受它们的独特魅力吧！😻 #德文卷毛猫 #萌宠日常
🐾德文卷毛猫
亲们～看这里！👀 超萌的德文卷毛猫来啦！🎉 它们毛茸茸的，性格超级温顺，每天都在猫舍里快乐成长哦！🏡 快来感受它们的独特魅力吧！😻 #德文卷毛猫 #萌宠日常
